# Lab 2 — The same assistant with LangChain (EP03)

In Lab 1 we wrote every line of the harness ourselves: tool schemas, `run_tool`, the loop, `history`, `MAX_STEPS`, `NEEDS_APPROVAL`.
In this lab we build **the same AI301 assistant** with **LangChain** (`create_agent`) and match it, piece by piece, to our own code

| LangChain | What we wrote ourselves in Lab 1 |
|---|---|
| `ChatOpenAI(...)` | `client` + `MODEL` + `EXTRA` + `call_llm` |
| `@tool` + type hints + docstring | the function + its JSON schema in `TOOLS` + `FUNCS` + `run_tool` |
| `create_agent(...)` | `run_agent()` — the perceive → reason → act loop |
| `checkpointer` + `thread_id` | the `history` list |
| `ModelCallLimitMiddleware` | `MAX_STEPS` |
| `HumanInTheLoopMiddleware` + `Command(resume=...)` | `NEEDS_APPROVAL` + `PENDING` + `approve()` |

The calendar and rooms are **the same simulated data as Lab 1**; every model answer comes from a real run.
(The video uses Thai prompts; this English notebook uses English ones, so your outputs will differ in wording.)

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, ch15 and Table 18.1 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **6 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep03-lab2-langchain-agent.en.ipynb`** (the same code as in the video).

## 0. Setup

You need Python 3.10+ (or Colab) and an **OpenRouter API key** — how to set up the key, and why we chose this model, is in Lab 1

The LangChain version is pinned, because framework APIs change quickly. With a newer version, some lines may need adjusting

In [ ]:
%pip install -q "langchain==1.4.7" "langgraph==1.2.16" "langchain-openai==1.7.2" "langchain-core==1.7.0" python-dotenv

In [ ]:
import os

# ---- from Lab 1: load the API key (Colab Secrets or a .env file) ----
def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

## 1. The model: `ChatOpenAI`

In Lab 1: `client` + `MODEL` + `EXTRA` and the `call_llm` function → in LangChain, a single object

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="google/gemma-4-26b-a4b-it", base_url="https://openrouter.ai/api/v1", api_key=load_key(),
                 extra_body={"reasoning": {"enabled": False},
                             "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}})

## 2. Tools: `@tool`

In Lab 1 we wrote each tool twice — the Python function and the JSON schema in `TOOLS` — then tied them together with `FUNCS` + `run_tool`.
In LangChain we write it once: `@tool` reads the **function name · type hints · docstring** and builds the schema for us

In [ ]:
# ---- from Lab 1: simulated data for teaching (today = 2026-11-10) ----
CALENDAR = {"today": "2026-11-10 (Tuesday)",
            "events": [{"name": "AI301 midterm exam", "date": "2026-11-18 (Wednesday)"},
                       {"name": "Homework 3 due", "date": "2026-11-14 (Saturday)"}]}
FREE = {"2026-11-12": {"Room 1": ["09:00"], "Room 3": ["13:00", "15:00"], "Room 5": []},
        "2026-11-13": {"Room 1": ["10:00", "13:00"], "Room 3": [], "Room 5": ["15:00"]}}
BOOKING = {}
SYSTEM = "You are the AI301 course assistant. Help students check the calendar and book tutoring rooms. Use tools to get real data; never guess dates."
QUESTION = "How many days until the AI301 midterm? And please book me a tutoring room this Thursday afternoon."

In [ ]:
# ✏️ TODO 1: Turn an ordinary function into a tool LangChain can hand to the model
from langchain_core.tools import tool

___
def get_calendar() -> dict:
    """Get today's date and the AI301 exam and due-date calendar"""
    return CALENDAR

@tool
def find_rooms(date: str) -> dict:
    """List tutoring rooms that are free on a given date (date as YYYY-MM-DD)"""
    if date not in FREE:
        return {"error": f"No room data for {date} (available: {', '.join(FREE)})"}
    return {"date": date, "free": {r: t for r, t in FREE[date].items() if t}}

@tool
def book_room(room: str, date: str, time: str) -> dict:
    """Book a tutoring room (one booking at a time; a new booking replaces the old one). date as YYYY-MM-DD, time as HH:MM"""
    if time not in FREE.get(date, {}).get(room, []):
        return {"ok": False, "error": f"{room} on {date} at {time} is not free"}
    BOOKING.clear(); BOOKING.update(room=room, date=date, time=time)
    return {"ok": True, "booking": dict(BOOKING)}

In [ ]:
TOOLS = [get_calendar, find_rooms, book_room]
print(book_room.name)
print(book_room.args)

## 3. The agent loop: `create_agent`

In Lab 1: `run_agent()`, about 20 lines → in LangChain: one function

In [ ]:
# ✏️ TODO 2: Create the agent from the model, the three tools and the system prompt
from langchain.agents import create_agent

agent = create_agent(llm, tools=___, system_prompt=SYSTEM)

In [ ]:
# ---- print helper: show only this turn's messages (like the prints in Lab 1's loop) ----
def show(result):
    msgs = result["messages"]
    last_user = max(i for i, m in enumerate(msgs) if m.type == "human")
    for m in msgs[last_user + 1:]:
        for c in getattr(m, "tool_calls", []):
            print(f"  🔧 {c['name']}({c['args']})")
        if m.type == "tool":
            print(f"  👀 {m.content}")
    if "__interrupt__" in result:
        for r in result["__interrupt__"][0].value["action_requests"]:
            print(f"⏸ waiting for approval: {r['name']}({r['args']})")
    else:
        print(msgs[-1].content)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
show(result)

In [ ]:
print([m.type for m in result["messages"]])

## 4. Persistence: `checkpointer` + `thread_id`

In Lab 1 the conversation continued because we passed the same `history` list in ourselves — so what about a second `invoke`?

In [ ]:
show(agent.invoke({"messages": [{"role": "user", "content": "13:00, please"}]}))
print(BOOKING)

In [ ]:
# ✏️ TODO 3: Give the agent a memory with a checkpointer
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
agent = create_agent(llm, tools=TOOLS, system_prompt=SYSTEM, checkpointer=___)

config = {"configurable": {"thread_id": "student-1"}}

In [ ]:
show(agent.invoke({"messages": [{"role": "user", "content": QUESTION}]}, config))

In [ ]:
# ✏️ TODO 4: Continue in the same thread: pass config to invoke
show(agent.invoke({"messages": [{"role": "user", "content": "13:00, please"}]}, ___))
print(BOOKING)

In [ ]:
show(agent.invoke({"messages": [{"role": "user", "content": "Can we move it to 3 pm?"}]},
                  {"configurable": {"thread_id": "student-2"}}))

## 5. Safety: middleware

**Middleware** is code plugged into the `create_agent` loop — before/after the model call, or before a tool runs

| Lab 1 | LangChain |
|---|---|
| `MAX_STEPS` | `ModelCallLimitMiddleware(run_limit=...)` |
| `NEEDS_APPROVAL` + `PENDING` | `HumanInTheLoopMiddleware(interrupt_on={...})` |
| `approve(history, ok=True)` | `invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)` |

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, HumanInTheLoopMiddleware
from langgraph.types import Command

limited = create_agent(llm, tools=TOOLS, system_prompt=SYSTEM,
                       middleware=[ModelCallLimitMiddleware(run_limit=1)])
show(limited.invoke({"messages": [{"role": "user", "content": QUESTION}]}))

In [ ]:
# ✏️ TODO 5: Make book_room wait for a person's approval before it runs
safe = create_agent(llm, tools=TOOLS, system_prompt=SYSTEM, checkpointer=memory,
                    middleware=[HumanInTheLoopMiddleware(interrupt_on=___)])

In [ ]:
show(safe.invoke({"messages": [{"role": "user", "content": "Can we move it to 3 pm?"}]}, config))
print(BOOKING)

In [ ]:
# ✏️ TODO 6: Approve the waiting request and let the agent continue
show(safe.invoke(Command(resume={"decisions": [{"type": ___}]}), config))
print(BOOKING)

## Summary: write it yourself vs a framework

| | Lab 1 (our own code) | Lab 2 (LangChain) |
|---|---|---|
| calling the model | `client` + `call_llm` | `ChatOpenAI` |
| tools | function + schema + `FUNCS` + `run_tool` | `@tool` |
| the agent loop | `run_agent()` | `create_agent()` |
| memory | the `history` list | `checkpointer` + `thread_id` |
| step limit | `MAX_STEPS` | `ModelCallLimitMiddleware` |
| human approval | `NEEDS_APPROVAL` + `approve()` | `HumanInTheLoopMiddleware` + `Command(resume=...)` |
| checking the room is free | in `book_room` | in `book_room` (still our job) |

The framework makes the code shorter and adds ready-made features (storing history in a database, reject/edit) — but every piece is something we could write ourselves in Lab 1.
When an agent behaves strangely, what you learned in Lab 1 is what tells you where to look

**Try next:** decline with `{"type": "reject", "message": "Make it Friday instead"}` and see what the agent does ·
add a `cancel_booking` tool and put it in `interrupt_on`

**Next video — the EP03 recap:** the agent loop, the harness and the 5 challenges, through what we saw in the two labs